In [3]:
import pandas as pd
import numpy as np
import psutil
from pathlib import Path

OUTPUT_DIR = Path(r"C:\mit_supercloud_raw")
RAW_FILE = r"C:\mit_supercloud_raw\nvidia_smi.csv"
CHUNKSIZE = 250_000

GROUP_KEYS = ["id_job", "Node", "gpu_index"]
USECOLS = GROUP_KEYS + ["temperature_gpu", "power_draw_W",
                        "utilization_gpu_pct", "utilization_memory_pct",
                        "memory_used_MiB"]

def ram_usage():
    mem = psutil.virtual_memory()
    return f"{mem.used/1e9:.1f}/{mem.total/1e9:.1f} GB ({mem.percent}%)"

def clean_keys(chunk):
    chunk["id_job"] = pd.to_numeric(chunk["id_job"], errors="coerce")
    chunk["gpu_index"] = pd.to_numeric(chunk["gpu_index"], errors="coerce")
    chunk["Node"] = chunk["Node"].astype(str).str.strip()
    chunk.dropna(subset=["id_job", "gpu_index"], inplace=True)
    chunk["id_job"] = chunk["id_job"].astype("int64")
    chunk["gpu_index"] = chunk["gpu_index"].astype("int64")
    return chunk

In [4]:
print("=" * 60)
print("STEP 1: aggregating raw dataset into job summary...")
print(f"   Initial RAM: {ram_usage()}")
print("=" * 60)

acc = {}  # (id_job, Node, gpu_index) -> running stats dict

for i, chunk in enumerate(pd.read_csv(RAW_FILE, usecols=USECOLS, chunksize=CHUNKSIZE, low_memory=False)):
    chunk = clean_keys(chunk)
    for col in ["temperature_gpu", "power_draw_W", "utilization_gpu_pct",
                "utilization_memory_pct", "memory_used_MiB"]:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    # aggregate WITHIN this chunk first (cheap, shrinks the merge step)
    partial = chunk.groupby(GROUP_KEYS).agg(
        peak_temp_c = ("temperature_gpu", "max"),
        sum_power   = ("power_draw_W", "sum"),   n_power = ("power_draw_W", "count"),
        sum_ug      = ("utilization_gpu_pct", "sum"),   n_ug = ("utilization_gpu_pct", "count"),
        sum_um      = ("utilization_memory_pct", "sum"), n_um = ("utilization_memory_pct", "count"),
        sum_mu      = ("memory_used_MiB", "sum"), n_mu = ("memory_used_MiB", "count"),
    )

    # merge into the running accumulator — same fix as your job_start_lookup logic,
    # so a job split across many chunks still ends up correct
    for key, row in partial.iterrows():
        if key not in acc:
            acc[key] = row.to_dict()
        else:
            a = acc[key]
            a["peak_temp_c"] = max(a["peak_temp_c"], row["peak_temp_c"])
            a["sum_power"] += row["sum_power"]; a["n_power"] += row["n_power"]
            a["sum_ug"]    += row["sum_ug"];    a["n_ug"]    += row["n_ug"]
            a["sum_um"]    += row["sum_um"];    a["n_um"]    += row["n_um"]
            a["sum_mu"]    += row["sum_mu"];    a["n_mu"]    += row["n_mu"]

    del chunk
    if (i + 1) % 100 == 0:
        print(f"  ✅ Chunk {i+1:>4} | {len(acc):,} job-GPU keys so far | RAM: {ram_usage()}")

print(f"✅ STEP 1 DONE. {len(acc):,} unique job-GPU records.")

STEP 1: aggregating raw dataset into job summary...
   Initial RAM: 11.9/16.4 GB (72.7%)
  ✅ Chunk  100 | 63 job-GPU keys so far | RAM: 11.9/16.4 GB (72.7%)
  ✅ Chunk  200 | 73 job-GPU keys so far | RAM: 11.9/16.4 GB (72.8%)
  ✅ Chunk  300 | 233 job-GPU keys so far | RAM: 12.1/16.4 GB (73.8%)
  ✅ Chunk  400 | 269 job-GPU keys so far | RAM: 12.1/16.4 GB (74.0%)
  ✅ Chunk  500 | 1,146 job-GPU keys so far | RAM: 11.9/16.4 GB (73.0%)
  ✅ Chunk  600 | 1,255 job-GPU keys so far | RAM: 12.0/16.4 GB (73.1%)
  ✅ Chunk  700 | 1,351 job-GPU keys so far | RAM: 12.0/16.4 GB (73.4%)
  ✅ Chunk  800 | 1,501 job-GPU keys so far | RAM: 12.1/16.4 GB (73.7%)
  ✅ Chunk  900 | 1,702 job-GPU keys so far | RAM: 12.0/16.4 GB (73.5%)
  ✅ Chunk 1000 | 1,834 job-GPU keys so far | RAM: 11.7/16.4 GB (71.2%)
  ✅ Chunk 1100 | 1,920 job-GPU keys so far | RAM: 11.7/16.4 GB (71.2%)
  ✅ Chunk 1200 | 1,952 job-GPU keys so far | RAM: 11.6/16.4 GB (71.0%)
  ✅ Chunk 1300 | 2,163 job-GPU keys so far | RAM: 11.6/16.4 GB (71.0%

In [5]:
records = []
for (id_job, node, gpu_index), a in acc.items():
    records.append({
        "id_job": id_job,
        "Node": node,
        "gpu_index": gpu_index,
        "peak_temp_c":  a["peak_temp_c"],
        "avg_power_w":  a["sum_power"] / a["n_power"],
        "avg_util_gpu": a["sum_ug"] / a["n_ug"],
        "avg_util_mem": a["sum_um"] / a["n_um"],
        "avg_mem_used": a["sum_mu"] / a["n_mu"],
    })

job_summary = pd.DataFrame.from_records(records)
job_summary.to_csv(OUTPUT_DIR / "job_summary.csv", index=False)

print(f"GPUs seen: {sorted(job_summary['gpu_index'].unique())}")
job_summary.head()

GPUs seen: [0, 1]


,id_job,Node,gpu_index,peak_temp_c,avg_power_w,avg_util_gpu,avg_util_mem,avg_mem_used
0,10330541552,r7419443-n948219,0,30.0,26.998936,0.0,0.0,3.903563
1,24627951701,r7753495-n303509,0,36.0,25.444748,0.0,0.0,3.797561
2,31824270204,r7753495-n303509,0,37.0,25.475123,0.0,0.0,3.889865
3,31830614139,r7753495-n303509,0,40.0,25.542672,0.0,0.0,3.382593
4,37761732261,r8141575-n303766,0,40.0,26.525073,0.0,0.0,3.997834


In [6]:
q1 = job_summary["peak_temp_c"].quantile(0.25)
q2 = job_summary["peak_temp_c"].quantile(0.50)
q3 = job_summary["peak_temp_c"].quantile(0.75)

print(f"Q1 = {q1:.2f}°C   Q2 = {q2:.2f}°C   Q3 = {q3:.2f}°C")

Q1 = 35.00°C   Q2 = 45.00°C   Q3 = 71.00°C


In [7]:
def region_of(t):
    if t < q1: return "Low"
    elif t < q2: return "Medium"
    elif t < q3: return "High"
    else: return "Critical"

job_summary["thermal_region"] = job_summary["peak_temp_c"].apply(region_of)
job_summary[["id_job", "Node", "gpu_index", "peak_temp_c", "thermal_region"]].head()

,id_job,Node,gpu_index,peak_temp_c,thermal_region
0,10330541552,r7419443-n948219,0,30.0,Low
1,24627951701,r7753495-n303509,0,36.0,Medium
2,31824270204,r7753495-n303509,0,37.0,Medium
3,31830614139,r7753495-n303509,0,40.0,Medium
4,37761732261,r8141575-n303766,0,40.0,Medium


In [8]:
region_order = ["Low", "Medium", "High", "Critical"]

region_power = (
    job_summary.groupby("thermal_region")["avg_power_w"]
    .mean()
    .reindex(region_order)
    .reset_index()
    .rename(columns={"avg_power_w": "region_avg_power_w"})
)
region_power

,thermal_region,region_avg_power_w
0,Low,27.937888
1,Medium,40.368655
2,High,91.618471
3,Critical,193.244077


In [9]:
MIN_PWM = 30  # set to None to disable clipping

critical_power = region_power.loc[
    region_power["thermal_region"] == "Critical", "region_avg_power_w"
].iloc[0]

pwm_table = region_power.copy()
pwm_table["relative_load"] = pwm_table["region_avg_power_w"] / critical_power
pwm_table["pwm_pct"] = (pwm_table["relative_load"] * 100).round(1)

if MIN_PWM is not None:
    pwm_table["pwm_pct_clipped"] = pwm_table["pwm_pct"].clip(lower=MIN_PWM)

pwm_table.to_csv(OUTPUT_DIR / "pwm_lookup_table.csv", index=False)
pwm_table

,thermal_region,region_avg_power_w,relative_load,pwm_pct,pwm_pct_clipped
0,Low,27.937888,0.144573,14.5,30.0
1,Medium,40.368655,0.208900,20.9,30.0
2,High,91.618471,0.474108,47.4,47.4
3,Critical,193.244077,1.000000,100.0,100.0
